# 02 · Caracterização de perfis e comparação econômica

**Objetivo:** caracterizar os registros retidos com movimentação, sem atribuir a movimentação à oferta ou inferir receita realizada.

**Fonte:** `data/raw/retention_case.xlsx` (não versionada); resultados agregados em `reports/private/profiles_summary.json`. **Unidade:** linha da base / episódio provisório.

**Definição principal:** `RETIDO` e pelo menos uma `FLAG_MOV_{BL,TV,FONE,MV}=1`; o indicador de movimentação *entre retidos* usa apenas registros `RETIDO` no denominador. Clientes podem aparecer mais de uma vez. Todos os agrupamentos abaixo têm pelo menos 100 observações; intervalos de Wilson não corrigem dependência entre episódios de um mesmo cliente.

**Execução:** `python -m retention_analytics.analysis_cli --input data/raw/retention_case.xlsx --out-dir reports/private` antes de abrir este notebook. Os outputs com métricas reais são mantidos somente no ambiente local.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
private = root / 'reports' / 'private'

profile = json.loads((private / 'profiles_summary.json').read_text(encoding='utf-8'))
profile['overall']

## Taxas e população de referência

Distinguir **população total**, **leads encerrados** e **retidos**. Não tratar leads abertos como insucessos. O material não fornece chaves de episódio nem acompanhamento pós-funil.

In [ ]:
total = profile['overall']['eligible']
ret = profile['overall']['retained']
movement = profile['overall']['retained_movement']
closed = ret + profile['overall']['not_retained']
print({'linhas_elegiveis': total, 'encerrados': closed,
       'retidos': ret, 'retidos_movimentacao': movement,
       'movimentacao_entre_retidos': movement / ret if ret else None,
       'evento_composto_entre_encerrados': movement / closed if closed else None})

## Mix inicial e movimentação entre os retidos

A comparação é *descritiva*. Diferenças entre grupos não representam o efeito causal de vender mais produtos.

In [ ]:
groups = sorted(
    [g for g in profile['segments'] if g['dimension'] == 'family_count_m0'
     and g['n_retained'] >= 100], key=lambda g: int(g['category'])
)
labels = [g['category'] for g in groups]
values = [100 * g['movement_among_retained'] for g in groups]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, values)
ax.set(title='Movimentação entre retidos por quantidade de famílias em M0',
       xlabel='Famílias contratadas em M0', ylabel='% dos registros retidos')
ax.set_ylim(0, 100)
plt.show()
[(g['category'], g['n_retained'], round(100*g['movement_among_retained'], 1)) for g in groups]

## Tempo e origem

As diferenças por origem podem refletir seleção do canal, mix de clientes e disponibilidade do desfecho. Analisar a proporção de `LEAD EM ABERTO` separadamente.

In [ ]:
for dimension in ('month', 'origin'):
    groups = sorted([g for g in profile['segments'] if g['dimension'] == dimension],
                    key=lambda x: x['category'])
    print(dimension, [
        (g['category'], g['n'], round(100*g['movement_among_retained'], 2)
         if g['movement_among_retained'] is not None else None) for g in groups
    ])

## Comparação econômica M0 versus M30

Os campos `VALOR_*` **não possuem unidade econômica e periodicidade validadas**. O delta abaixo é apenas a variação dos valores registrados por família entre snapshots, não receita realizada nem uplift causal. Agregação de bundles pode superestimar ou subestimar valores.

In [ ]:
economic = profile['value_snapshot_comparison']
for name, data in economic.items():
    print(name, {'n': data['n'], 'media_delta': round(data['mean_change'], 2),
                'mediana_delta': data['median_change'],
                'percentual_delta_negativo': round(100 * data['share_negative'], 1),
                'variacao_soma_valores': round(100 * data['relative_sum_change'], 2)
                if data['relative_sum_change'] is not None else None})

## Limitações e próximo uso

O estudo considera registros, não uma amostra probabilística garantida de clientes únicos. Todas as associações dependem de validação da chave de episódio, da semântica de `RETIDO` e `FLAG_MOV_*`, da origem da amostra e da unidade monetária. Os resultados desta etapa informam hipóteses para experimentos, não recomendação causal de oferta.